<a href="https://colab.research.google.com/github/DivyaMeenaSundaram/Prompt-Engineering/blob/main/Costomer_support_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip -q install streamlit google-genai langsmith pyngrok

In [41]:
GEMINI_API_KEY = "your key"

LANGSMITH_API_KEY = "your key"


# Name of the LangSmith project
LANGSMITH_PROJECT = "customer-support-prompt-lab"


# Gemini model
GEMINI_MODEL = "gemini-3.6-flash"


# ============================================================
# LOAD THE SETTINGS
# ============================================================

import os

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

os.environ["LANGSMITH_TRACING"] = "true"

os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT


# ============================================================
# CHECK
# ============================================================

print(
    "Gemini key loaded:",
    bool(GEMINI_API_KEY) and
    "PASTE_" not in GEMINI_API_KEY
)

print(
    "LangSmith key loaded:",
    bool(LANGSMITH_API_KEY) and
    "PASTE_" not in LANGSMITH_API_KEY
)

Gemini key loaded: True
LangSmith key loaded: True


In [42]:
# ============================================================
# STEP 3: TEST GEMINI API
# ============================================================

# Import Google's Gemini library
from google import genai

client = genai.Client(
    api_key=GEMINI_API_KEY
)

# ------------------------------------------------------------
# Give Gemini a simple test question
# ------------------------------------------------------------

test_question = """
A customer says:

"My order has not arrived yet.
What should I do?"

Give a short customer-support response.
"""

# ------------------------------------------------------------
# Send the question to Gemini
# ------------------------------------------------------------

response = client.models.generate_content(

    model=GEMINI_MODEL,

    contents=test_question
)

# ------------------------------------------------------------
# Display Gemini's response
# ------------------------------------------------------------


print("GEMINI API TEST")
print()
print(response.text)
print()


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 45.585336889s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.6-flash', 'location': 'global'},

In [ ]:
# ============================================================
# STEP 5: LAUNCH THE STREAMLIT APPLICATION
# ============================================================
#
# This starts our chatbot application from Google Colab.
#
# You do NOT need:
# - VS Code
# - Command Prompt
# - Terminal
# - A local folder
#
# Colab will run everything for us.
# ============================================================


# ------------------------------------------------------------
# Import the tools we need
# ------------------------------------------------------------

import subprocess
import time
import requests


# ------------------------------------------------------------
# Start Streamlit
# ------------------------------------------------------------

process = subprocess.Popen(

    [
        "streamlit",
        "run",
        "app.py",

        # Streamlit configuration
        "--server.port",
        "8501",

        "--server.address",
        "0.0.0.0",

        "--server.headless",
        "true"

    ],

    stdout=subprocess.PIPE,

    stderr=subprocess.PIPE

)


# Give Streamlit a few seconds to start
time.sleep(5)


# ------------------------------------------------------------
# Check whether Streamlit is running
# ------------------------------------------------------------

try:

    response = requests.get(

        "http://localhost:8501"

    )

    print(
        "Streamlit server started successfully."
    )

    print(
        "Status code:",
        response.status_code
    )

except Exception as e:

    print(
        "Streamlit may still be starting..."
    )

    print(
        "Error:",
        e
    )


# ------------------------------------------------------------
# Create a public URL using Colab's proxy
# ------------------------------------------------------------
#
# Google Colab provides a way to access applications
# running inside the notebook.
# ------------------------------------------------------------

from google.colab.output import eval_js

public_url = eval_js(
    "google.colab.kernel.proxyPort(8501)"
)


# ------------------------------------------------------------
# Display the link
# ------------------------------------------------------------

print()
print("==========================================")
print("🎉 CUSTOMER SUPPORT AI IS LIVE")
print("==========================================")
print()
print("OPEN YOUR APPLICATION:")
print()
print(public_url)
print()
print("==========================================")

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

app_path = Path("/content/app.py")

print("app.py exists:", app_path.exists())
print("File size:", app_path.stat().st_size, "bytes")

app.py exists: True
File size: 15819 bytes


In [ ]:
from pathlib import Path

app_path = Path("/content/app.py")
text = app_path.read_text()

# ============================================================
# SAFETY CHECK
# ============================================================

if 'with tab_eval:' not in text:
    raise RuntimeError(
        "Could not find the Evaluation tab in app.py. "
        "Do not continue; the original app structure is different."
    )

# ============================================================
# FIND THE EVALUATION TAB
# ============================================================

eval_start = text.index('with tab_eval:')

# Find the next major divider after the evaluation section.
eval_end = text.find('st.divider()', eval_start)

if eval_end == -1:
    raise RuntimeError(
        "Could not determine the end of the Evaluation tab."
    )

# ============================================================
# NEW EVALUATION TAB
# ============================================================

new_eval = r"""
with tab_eval:

    st.subheader("📊 Prompt Evaluation")

    st.write(
        "Compare two prompt versions using the same customer questions. "
        "The evaluation measures policy correctness, relevance, "
        "groundedness, escalation behaviour, and conciseness."
    )

    st.divider()

    # --------------------------------------------------------
    # A/B PROMPT SELECTION
    # --------------------------------------------------------

    st.markdown("### 🧪 A/B Prompt Experiment")

    col1, col2 = st.columns(2)

    prompt_names = list(PROMPTS.keys())

    with col1:
        version_a = st.selectbox(
            "Prompt A",
            prompt_names,
            index=min(2, len(prompt_names) - 1),
            key="eval_version_a"
        )

    with col2:
        version_b = st.selectbox(
            "Prompt B",
            prompt_names,
            index=min(3, len(prompt_names) - 1),
            key="eval_version_b"
        )

    st.info(
        "Both prompt versions receive exactly the same five questions. "
        "This makes the test an A/B prompt comparison."
    )

    # --------------------------------------------------------
    # DATASET
    # --------------------------------------------------------

    with st.expander("View evaluation questions"):

        for i, question in enumerate(SAMPLE_QUESTIONS, 1):
            st.write(f"**Q{i}.** {question}")

    # --------------------------------------------------------
    # LOCAL EVALUATION FUNCTION
    # --------------------------------------------------------

    def local_evaluate(question, answer):

        q = question.lower()
        a = answer.lower()

        accuracy = True
        relevance = True
        groundedness = True
        escalation = True
        concise = True

        # Accuracy / policy correctness
        if "where is my order" in q:

            accuracy = (
                "order" in a
                and (
                    "3–5 business days" in a
                    or "3-5 business days" in a
                    or "business days" in a
                    or "delivery" in a
                )
            )

            relevance = "order" in a or "delivery" in a

        elif "delivery address" in q:

            accuracy = (
                "address" in a
                and (
                    "dispatch" in a
                    or "before" in a
                )
            )

            relevance = "address" in a

        elif "payment failed" in q:

            accuracy = "payment" in a

            relevance = (
                "payment" in a
                or "charged" in a
                or "support" in a
            )

            escalation = (
                "escalat" in a
                or "human support" in a
                or "support team" in a
                or "representative" in a
            )

        elif "refund" in q:

            accuracy = (
                "refund" in a
                and (
                    "human" in a
                    or "support" in a
                    or "review" in a
                )
            )

            relevance = "refund" in a

            escalation = (
                "human" in a
                or "support" in a
                or "representative" in a
            )

        elif "wrong product" in q:

            accuracy = (
                "order" in a
                or "order id" in a
                or "support" in a
            )

            relevance = (
                "wrong" in a
                or "product" in a
            )

            escalation = (
                "support" in a
                or "representative" in a
                or "contact" in a
            )

        # ----------------------------------------------------
        # Groundedness / hallucination detection
        # ----------------------------------------------------

        hallucination_phrases = [
            "i checked your order",
            "i checked the system",
            "your order is currently at",
            "your package is currently at",
            "your order will arrive tomorrow",
            "your order will arrive on",
            "i can see your order",
            "your tracking shows",
            "tracking number is",
            "guaranteed delivery",
            "definitely arrive"
        ]

        for phrase in hallucination_phrases:

            if phrase in a:
                groundedness = False

        # ----------------------------------------------------
        # Conciseness
        # ----------------------------------------------------

        word_count = len(answer.split())

        concise = word_count <= 100

        # ----------------------------------------------------
        # Overall score
        # ----------------------------------------------------

        values = [
            accuracy,
            relevance,
            groundedness,
            escalation,
            concise
        ]

        score = round(
            sum(1 for value in values if value)
            / len(values)
            * 100
        )

        return {
            "Accuracy": accuracy,
            "Relevance": relevance,
            "Groundedness": groundedness,
            "Escalation": escalation,
            "Conciseness": concise,
            "Word Count": word_count,
            "Score": score
        }

    # --------------------------------------------------------
    # RUN ONE PROMPT VERSION
    # --------------------------------------------------------

    def run_eval_version(version_name):

        results = []

        for question in SAMPLE_QUESTIONS:

            try:

                answer, prompt_used = traced_generate(
                    question,
                    version_name
                )

                metrics = local_evaluate(
                    question,
                    answer
                )

                results.append({
                    "Question": question,
                    "Answer": answer,
                    "Metrics": metrics,
                    "Error": None
                })

            except Exception as e:

                results.append({
                    "Question": question,
                    "Answer": "",
                    "Metrics": None,
                    "Error": str(e)
                })

        return results

    # --------------------------------------------------------
    # RUN A/B TEST
    # --------------------------------------------------------

    if st.button(
        "▶ Run A/B Prompt Evaluation",
        type="primary",
        use_container_width=True
    ):

        if not gemini_key:

            st.error(
                "Enter your Gemini API key in the sidebar."
            )

        else:

            with st.spinner(
                "Running the A/B prompt experiment..."
            ):

                results_a = run_eval_version(version_a)

                results_b = run_eval_version(version_b)

            st.session_state.eval_ab_results = {
                "A": results_a,
                "B": results_b,
                "version_a": version_a,
                "version_b": version_b
            }

            st.success(
                "A/B prompt evaluation completed."
            )

    # --------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------

    if "eval_ab_results" in st.session_state:

        data = st.session_state.eval_ab_results

        if data is not None:

            results_a = data["A"]
            results_b = data["B"]

            version_a = data["version_a"]
            version_b = data["version_b"]

            name_a = PROMPTS[version_a]["version"]
            name_b = PROMPTS[version_b]["version"]

            # ------------------------------------------------
            # METRIC CALCULATION
            # ------------------------------------------------

            metric_names = [
                "Accuracy",
                "Relevance",
                "Groundedness",
                "Escalation",
                "Conciseness"
            ]

            def metric_percentage(results, metric):

                values = []

                for result in results:

                    if result["Metrics"] is not None:
                        values.append(
                            bool(
                                result["Metrics"][metric]
                            )
                        )

                if len(values) == 0:
                    return 0

                return round(
                    sum(values) / len(values) * 100
                )

            def overall_percentage(results):

                scores = []

                for result in results:

                    if result["Metrics"] is not None:
                        scores.append(
                            result["Metrics"]["Score"]
                        )

                if len(scores) == 0:
                    return 0

                return round(
                    sum(scores) / len(scores)
                )

            # ------------------------------------------------
            # OVERALL COMPARISON
            # ------------------------------------------------

            st.divider()

            st.markdown(
                "### 📈 Overall A/B Comparison"
            )

            comparison = []

            for metric in metric_names:

                comparison.append({
                    "Metric": metric,
                    name_a: (
                        f"{metric_percentage(results_a, metric)}%"
                    ),
                    name_b: (
                        f"{metric_percentage(results_b, metric)}%"
                    )
                })

            comparison.append({
                "Metric": "OVERALL SCORE",
                name_a: f"{overall_percentage(results_a)}%",
                name_b: f"{overall_percentage(results_b)}%"
            })

            st.dataframe(
                comparison,
                use_container_width=True,
                hide_index=True
            )

            # ------------------------------------------------
            # METRIC BARS
            # ------------------------------------------------

            st.markdown(
                "### 📊 Metric-by-Metric Results"
            )

            for metric in metric_names:

                score_a = metric_percentage(
                    results_a,
                    metric
                )

                score_b = metric_percentage(
                    results_b,
                    metric
                )

                st.write(f"**{metric}**")

                ca, cb = st.columns(2)

                with ca:

                    st.caption(name_a)

                    st.progress(
                        score_a / 100
                    )

                    st.write(
                        f"{score_a}%"
                    )

                with cb:

                    st.caption(name_b)

                    st.progress(
                        score_b / 100
                    )

                    st.write(
                        f"{score_b}%"
                    )

            # ------------------------------------------------
            # QUESTION LEVEL COMPARISON
            # ------------------------------------------------

            st.divider()

            st.markdown(
                "### 🔬 Question-by-Question Comparison"
            )

            for i, question in enumerate(
                SAMPLE_QUESTIONS
            ):

                st.markdown(
                    f"#### Q{i + 1}. {question}"
                )

                ca, cb = st.columns(2)

                result_a = results_a[i]
                result_b = results_b[i]

                with ca:

                    st.markdown(
                        f"**{name_a}**"
                    )

                    if result_a["Error"]:

                        st.error(
                            "Gemini request failed."
                        )

                    else:

                        st.write(
                            result_a["Answer"]
                        )

                        st.caption(
                            "Score: "
                            + str(
                                result_a["Metrics"]["Score"]
                            )
                            + "%"
                        )

                with cb:

                    st.markdown(
                        f"**{name_b}**"
                    )

                    if result_b["Error"]:

                        st.error(
                            "Gemini request failed."
                        )

                    else:

                        st.write(
                            result_b["Answer"]
                        )

                        st.caption(
                            "Score: "
                            + str(
                                result_b["Metrics"]["Score"]
                            )
                            + "%"
                        )

            # ------------------------------------------------
            # TEACHING EXPLANATION
            # ------------------------------------------------

            st.divider()

            st.markdown(
                "### 🎓 What does this evaluation demonstrate?"
            )

            st.write(
                "The same customer questions are given to two "
                "different prompt versions. Their responses are "
                "then evaluated using multiple criteria."
            )

            st.write(
                "This allows prompt engineering to be treated "
                "as an experiment rather than simply judging "
                "whether one response looks better."
            )

            st.write(
                "The A/B comparison can reveal whether a prompt "
                "change improves policy correctness, relevance, "
                "groundedness, escalation behaviour, or "
                "conciseness."
            )
"""

# ============================================================
# INSERT THE NEW TAB
# ============================================================

text = (
    text[:eval_start]
    + new_eval
    + "\n\n"
    + text[eval_end:]
)

# ============================================================
# SAVE
# ============================================================

app_path.write_text(text)

print("==============================================")
print("✓ Evaluation tab replaced")
print("✓ A/B testing added")
print("✓ Accuracy added")
print("✓ Relevance added")
print("✓ Groundedness added")
print("✓ Escalation added")
print("✓ Conciseness added")
print("✓ Overall score added")
print("✓ Question-level comparison added")
print("✓ Existing Live Chat preserved")
print("✓ Existing Prompt Template preserved")
print("✓ Existing Version History preserved")
print("✓ LangSmith tracing preserved")
print("==============================================")

✓ Evaluation tab replaced
✓ A/B testing added
✓ Accuracy added
✓ Relevance added
✓ Groundedness added
✓ Escalation added
✓ Conciseness added
✓ Overall score added
✓ Question-level comparison added
✓ Existing Live Chat preserved
✓ Existing Prompt Template preserved
✓ Existing Version History preserved
✓ LangSmith tracing preserved


In [ ]:
import py_compile

try:
    py_compile.compile(
        "/content/app.py",
        doraise=True
    )
    print("✓ app.py syntax is correct")
except Exception as e:
    print("❌ Syntax error:")
    print(e)

✓ app.py syntax is correct


In [ ]:
!pkill -f streamlit

import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--server.enableWebsocketCompression=false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(8)

print("✓ Streamlit restarted")

✓ Streamlit restarted


In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")

print(url)

https://8501-m-s-kkb-ase1a0-9kadxdjb091h-a.asia-east1-0.prod.colab.dev
